# Celesta — Kepler Exoplanet Classifier
## India High School Exoplanet Data Challenge
### Google Colab Notebook — runs entirely in the cloud, no local setup needed

**Author:** Srinath V Venkateshan  
**Contact:** vvsrinath0@gmail.com

---

### What this notebook does

This notebook walks through the complete Celesta ML pipeline from scratch:

1. Downloads the NASA Kepler KOI dataset directly from the NASA Exoplanet Archive
2. Explores the data — distributions, missing values, class imbalance, correlations
3. Explains every feature in plain English
4. Engineers 8 new physics-motivated features from the raw measurements
5. Trains a soft-voting ensemble (XGBoost + HistGradientBoosting + Random Forest)
6. Evaluates with confusion matrix, ROC curves, and 5-fold cross-validation
7. Explains predictions with SHAP values
8. Discusses what the model learned and why it works

**Runtime:** ~10 minutes on a free Colab T4 GPU (training is CPU-only but fast enough)

---

### How to use this notebook

1. Open in [Google Colab](https://colab.research.google.com)
2. Go to **Runtime → Run all** (or press Ctrl+F9)
3. That's it — everything downloads and runs automatically

---

### Table of Contents
- [Section 1: Setup](#section-1)
- [Section 2: The Problem — What Are We Classifying?](#section-2)
- [Section 3: Data Download and Loading](#section-3)
- [Section 4: Exploratory Data Analysis](#section-4)
- [Section 5: Missing Values and Data Cleaning](#section-5)
- [Section 6: Feature Engineering](#section-6)
- [Section 7: Model Architecture](#section-7)
- [Section 8: Training](#section-8)
- [Section 9: Evaluation](#section-9)
- [Section 10: Cross-Validation](#section-10)
- [Section 11: SHAP Explainability](#section-11)
- [Section 12: What the Model Learned](#section-12)
- [Section 13: Conclusion](#section-13)

---
## Section 1: Setup

Install the packages we need beyond what Colab already has.

In [ ]:
# Install packages not pre-installed in Colab
!pip install -q xgboost shap

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap

from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

# Dark space-themed plot style to match the Celesta web app
plt.rcParams.update({
    'figure.facecolor':  '#0d1117',
    'axes.facecolor':    '#161b22',
    'axes.edgecolor':    '#30363d',
    'axes.labelcolor':   '#e6edf3',
    'xtick.color':       '#8b949e',
    'ytick.color':       '#8b949e',
    'text.color':        '#e6edf3',
    'grid.color':        '#21262d',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'sans-serif',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
})

# Colour palette — one colour per class
CLASS_COLORS = {
    'CONFIRMED':      '#4fc3f7',
    'CANDIDATE':      '#a5d6a7',
    'FALSE POSITIVE': '#ef9a9a',
}

print('Setup complete.')

---
## Section 2: The Problem — What Are We Classifying?

### Background: The Kepler Space Telescope

NASA's Kepler telescope (2009–2018) monitored ~150,000 stars and searched for planets using the **transit method**: when a planet orbits its star, it periodically crosses in front of it from our line of sight, causing a tiny but measurable dip in the star's brightness.

Kepler detected thousands of these dips, but not all of them are planets. Many are caused by:

- **Eclipsing binary stars** — two stars orbiting each other, one dimming the other
- **Background eclipsing binaries** — a distant binary that happens to sit behind the target star
- **Instrumental noise** — electronic artifacts, cosmic rays, detector crosstalk
- **Grazing transits** — a stellar companion skimming the limb of the star

### The Classification Task

Every transit-like signal is recorded as a **Kepler Object of Interest (KOI)** and eventually labelled:

| Label | What it means |
|---|---|
| **CONFIRMED** | Follow-up observations (spectroscopy, RV) proved it's a real planet |
| **CANDIDATE** | Passed automated checks but awaits confirmation |
| **FALSE POSITIVE** | Shown by follow-up to NOT be a planet |

Our job: **predict the label from raw telescope measurements, with zero data leakage**.

### Why Leakage Matters

The NASA archive includes `koi_pdisposition` (NASA's own pre-classification) and `koi_score` (NASA's confidence). Using those would give us 99%+ accuracy — but we'd just be copying NASA's answer. That tells us nothing about the underlying astronomy and would be useless in a real survey.

We use **only raw transit photometry and stellar parameters** — the same data a real astrophysicist would start with before any human review.

---
## Section 3: Data Download and Loading

We download directly from the NASA Exoplanet Archive API. No account or API key needed — it's fully public.

In [ ]:
import urllib.request

# NASA Exoplanet Archive — KOI Cumulative Table, CSV format
# The select= parameter grabs only the columns we care about, keeping the download fast
NASA_URL = (
    "https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI"
    "?table=cumulative&select=*&format=csv"
)

print("Downloading KOI Cumulative Table from NASA Exoplanet Archive...")
print("(This may take 30-60 seconds depending on NASA server load)")

try:
    urllib.request.urlretrieve(NASA_URL, "koi_cumulative.csv")
    print("Download complete.")
except Exception as e:
    print(f"Direct download failed ({e})")
    print("Trying alternative URL...")
    # Alternative: TAP service
    ALT_URL = (
        "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
        "?query=select+*+from+cumulative&format=csv"
    )
    urllib.request.urlretrieve(ALT_URL, "koi_cumulative.csv")
    print("Alternative download complete.")

In [ ]:
raw = pd.read_csv("koi_cumulative.csv", comment='#')
print(f"Raw dataset: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"\nColumn names (first 40):")
print(list(raw.columns[:40]))

In [ ]:
# These are the raw Kepler measurements we allow the model to use
# koi_sage, koi_model_dof, koi_model_chisq are intentionally excluded — 100% missing
# All koi_pdisposition, koi_score, koi_vet_* are excluded — they would be data leakage

RAW_FEATURES = [
    # Transit shape
    'koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration',
    'koi_depth', 'koi_ror', 'koi_srho', 'koi_prad', 'koi_sma',
    'koi_incl', 'koi_teq', 'koi_insol', 'koi_dor',
    # Signal statistics
    'koi_max_sngle_ev', 'koi_max_mult_ev', 'koi_model_snr',
    'koi_count', 'koi_num_transits', 'koi_bin_oedp_sig',
    # Stellar properties
    'koi_steff', 'koi_slogg', 'koi_smet', 'koi_srad', 'koi_smass',
    # Sky position and brightness
    'ra', 'dec', 'koi_kepmag',
]

TARGET = 'koi_disposition'

available = [c for c in RAW_FEATURES if c in raw.columns]
print(f"Available raw features: {len(available)} / {len(RAW_FEATURES)}")

df = raw[available + [TARGET]].copy()
df = df[df[TARGET].notna()].copy()

print(f"\nAfter removing rows with missing target: {df.shape[0]:,} rows")
print(f"\nClass distribution:")
print(df[TARGET].value_counts())

---
## Section 4: Exploratory Data Analysis

Before training anything, we need to understand the data — its shape, its distributions, and what separates the classes visually.

In [ ]:
# ── 4.1 Class imbalance ───────────────────────────────────────────────────────
counts = df[TARGET].value_counts()
total = len(df)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [CLASS_COLORS[c] for c in counts.index]
bars = ax.barh(counts.index, counts.values, color=colors, edgecolor='none', height=0.5)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{count:,}  ({count/total*100:.1f}%)',
            va='center', fontsize=10, color='#e6edf3')

ax.set_xlabel('Number of KOIs')
ax.set_title('Class Distribution — Kepler Objects of Interest', pad=12)
ax.set_xlim(0, counts.max() * 1.25)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("\nKey observation: FALSE POSITIVE outnumbers CANDIDATE by {:.1f}x".format(
    counts['FALSE POSITIVE'] / counts['CANDIDATE']))
print("This class imbalance must be handled explicitly in training.")

In [ ]:
# ── 4.2 Feature distributions by class ───────────────────────────────────────
# These are the features that show the clearest separation between classes

key_features = [
    ('koi_model_snr',    'Transit SNR'),
    ('koi_max_mult_ev',  'Multi-Event Statistic'),
    ('koi_prad',         'Planet Radius (R⊕)'),
    ('koi_period',       'Orbital Period (days)'),
    ('koi_impact',       'Impact Parameter'),
    ('koi_depth',        'Transit Depth (ppm)'),
    ('koi_ror',          'Planet/Star Radius Ratio'),
    ('koi_bin_oedp_sig', 'Odd-Even Depth Significance'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (feat, label) in enumerate(key_features):
    ax = axes[i]
    for cls, color in CLASS_COLORS.items():
        vals = df[df[TARGET] == cls][feat].dropna()
        # log transform for display — these are all right-skewed
        vals_log = np.log1p(vals.clip(lower=0))
        ax.hist(vals_log, bins=45, alpha=0.55, color=color, label=cls, density=True)
    ax.set_title(f'log(1 + {label})', fontsize=9)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7, loc='upper right')

fig.suptitle('Feature Distributions by Class (log scale)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Observations:")
print("- CONFIRMED planets (blue) cluster at higher SNR and multi-event stat")
print("- FALSE POSITIVES (red) spread more broadly — they are diverse in nature")
print("- CANDIDATES (green) overlap with both — they are genuinely ambiguous by definition")

In [ ]:
# ── 4.3 SNR vs planet radius scatter — the most separating 2D view ───────────
sample = df.dropna(subset=['koi_model_snr', 'koi_prad', TARGET])

fig, ax = plt.subplots(figsize=(10, 6))
for cls, color in CLASS_COLORS.items():
    sub = sample[sample[TARGET] == cls]
    ax.scatter(
        np.log1p(sub['koi_prad'].clip(lower=0)),
        np.log1p(sub['koi_model_snr'].clip(lower=0)),
        alpha=0.2, s=8, color=color, label=cls
    )

ax.set_xlabel('log(1 + Planet Radius)  [R⊕]', fontsize=11)
ax.set_ylabel('log(1 + Transit SNR)', fontsize=11)
ax.set_title('Planet Radius vs Transit SNR — coloured by disposition', pad=12)
legend = ax.legend(markerscale=3, framealpha=0.2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Pattern: CONFIRMED planets (blue) cluster at small-medium radius and high SNR.")
print("Large-radius, low-SNR objects are almost always FALSE POSITIVEs.")

In [ ]:
# ── 4.4 Correlation matrix ────────────────────────────────────────────────────
top_feats = [
    'koi_model_snr', 'koi_max_mult_ev', 'koi_ror', 'koi_prad',
    'koi_dor', 'koi_impact', 'koi_depth', 'koi_period',
    'koi_teq', 'koi_max_sngle_ev',
]
corr = df[top_feats].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax,
    annot_kws={'size': 8}, linewidths=0.4,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Pairwise Feature Correlations', pad=12)
plt.tight_layout()
plt.show()

print("Strong correlations to note:")
print("- koi_prad and koi_ror: both measure planet size (radius directly vs ratio to star)")
print("- koi_model_snr and koi_max_mult_ev: both measure overall signal quality")
print("These redundancies help the ensemble — RF uses one, XGBoost uses the other, averaging out noise")

---
## Section 5: Missing Values and Data Cleaning

Real-world data always has gaps. Understanding *why* values are missing is as important as deciding how to handle them.

In [ ]:
miss = df[available].isnull().mean().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#ef9a9a' if v > 0.5 else '#ffcc80' if v > 0.1 else '#a5d6a7'
              for v in miss_nonzero.values]
ax.barh(miss_nonzero.index, miss_nonzero.values * 100, color=bar_colors)
ax.axvline(10, color='#ffcc80', linewidth=1.2, linestyle='--', label='10% threshold')
ax.set_xlabel('Missing values (%)')
ax.set_title('Missing Values by Feature', pad=12)
legend_elements = [
    mpatches.Patch(color='#ef9a9a', label='>50% missing'),
    mpatches.Patch(color='#ffcc80', label='10-50% missing'),
    mpatches.Patch(color='#a5d6a7', label='<10% missing'),
]
ax.legend(handles=legend_elements)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nStrategy per missing level:")
print("  <10% missing  → XGBoost and HGB handle NaN natively; RF uses median imputation")
print("  >50% missing  → would have already been dropped (koi_sage, koi_model_dof, koi_model_chisq)")
print()
print("Why median imputation for RF (not mean)?")
print("Features like koi_model_snr are right-skewed. The mean is pulled toward outliers.")
print("Median is a more robust centre for skewed distributions.")

---
## Section 6: Feature Engineering

Raw features describe *what Kepler measured*. Engineered features describe *what those measurements mean physically*. The goal is to give the model concepts that a human astrophysicist would naturally use when evaluating a KOI.

We add 8 derived features, each motivated by specific astrophysical reasoning.

In [ ]:
def add_features(df):
    d = df.copy()
    eps = 1e-9  # avoid division by zero

    # ── Feature 1: single_multi_ratio ─────────────────────────────────────────
    # Single-event / multi-event statistic.
    # A real repeating transit: multi-event stat grows as sqrt(N) with transit count.
    # A one-off event (cosmic ray, background binary flare): single-event is high
    # but multi-event barely improves because there's no consistent repeating signal.
    # High ratio = likely false positive.
    d['single_multi_ratio'] = d['koi_max_sngle_ev'] / (d['koi_max_mult_ev'] + eps)

    # ── Feature 2: duration_period_ratio ──────────────────────────────────────
    # Transit duration / orbital period (converted to hours).
    # For circular orbits: this ratio encodes the transit chord length,
    # which depends on the stellar radius and orbital semi-major axis.
    # Via Kepler's Third Law (P² ∝ a³/M), it's a stellar density proxy.
    # Inconsistent density (compared to stellar spectra) flags false positives.
    d['duration_period_ratio'] = d['koi_duration'] / (d['koi_period'] * 24.0 + eps)

    # ── Features 3-5: log transforms ─────────────────────────────────────────
    # Orbital periods range from 0.2 to 600+ days (3 orders of magnitude).
    # Transit depths range from 10 to 100,000+ ppm.
    # These are log-normally distributed. Log transform spreads the information
    # more evenly and makes decision boundaries easier for trees to find.
    d['log_period'] = np.log1p(d['koi_period'].clip(lower=0))
    d['log_depth']  = np.log1p(d['koi_depth'].clip(lower=0))
    d['log_snr']    = np.log1p(d['koi_model_snr'].clip(lower=0))

    # ── Feature 6: stellar_density_proxy ─────────────────────────────────────
    # M_star / R_star³ is proportional to mean stellar density.
    # Dense M-dwarf stars: a transiting large object is implausible (it would be
    # bigger than the star). A high-density star with a large-radius KOI is FP.
    d['stellar_density_proxy'] = d['koi_smass'] / (d['koi_srad'] ** 3 + eps)

    # ── Feature 7: impact_ror_ratio ───────────────────────────────────────────
    # Impact parameter b / planet-star radius ratio (Rp/Rs).
    # b = 0: central crossing. b = 1: grazing the stellar limb.
    # When b / (Rp/Rs) ≈ 1: the companion barely clips the star edge.
    # Grazing transits are common in eclipsing binary false positives:
    # a bright companion at high inclination mimics a small planet depth.
    d['impact_ror_ratio'] = d['koi_impact'] / (d['koi_ror'] + eps)

    # ── Feature 8: expected_duration_ratio ───────────────────────────────────
    # Kepler geometry predicts transit duration as:
    #   τ_expected ∝ (P/π) × (Rp/Rs) / (a/Rs) × sqrt(1 - b²)
    # If observed duration ≠ expected: the orbit may be eccentric (planet speeds
    # up at perihelion, shortening transit) OR it's a background binary with
    # different orbital parameters than assumed.
    expected = (
        d['koi_period'] * 24.0 / np.pi
        * d['koi_ror'] / (d['koi_dor'] + eps)
        * np.sqrt((1 - d['koi_impact'] ** 2).clip(lower=0))
    )
    d['expected_duration_ratio'] = d['koi_duration'] / (expected + eps)

    return d


ENGINEERED = [
    'single_multi_ratio', 'duration_period_ratio',
    'log_period', 'log_depth', 'log_snr',
    'stellar_density_proxy', 'impact_ror_ratio', 'expected_duration_ratio',
]

df = add_features(df)
ALL_FEATURES = available + ENGINEERED

print(f"Raw features:        {len(available)}")
print(f"Engineered features: {len(ENGINEERED)}")
print(f"Total:               {len(ALL_FEATURES)}")

In [ ]:
# Visualise the two most powerful engineered features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (feat, title, clip_pct) in zip(axes, [
    ('single_multi_ratio',      'single_multi_ratio\n(Single ÷ Multi Event Stat)', 99),
    ('expected_duration_ratio', 'expected_duration_ratio\n(Observed ÷ Predicted Duration)', 99),
]):
    for cls, color in CLASS_COLORS.items():
        vals = df[df[TARGET] == cls][feat].dropna()
        upper = np.percentile(vals.clip(lower=0), clip_pct)
        vals_clipped = np.log1p(vals.clip(lower=0, upper=upper))
        ax.hist(vals_clipped, bins=50, alpha=0.55, color=color, label=cls, density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('log(1 + value)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Engineered Features — Class Separation', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("single_multi_ratio separates FALSE POSITIVEs from CONFIRMEDs clearly.")
print("expected_duration_ratio catches eccentric orbits and blended binaries.")

---
## Section 7: Model Architecture

### Why a Soft-Voting Ensemble?

No single algorithm is best at everything on this dataset:

| Estimator | Strength | Weakness |
|---|---|---|
| **XGBoost** | Best at complex non-linear interactions; native NaN handling | Can overfit without careful tuning |
| **HistGradBoost** | Fast, handles NaN, respects class weights | Similar inductive bias to XGBoost |
| **Random Forest** | High variance, low correlation with boosters | Needs imputer; slower |

A **soft-voting ensemble** averages the probability predictions. When models disagree, the average is more conservative. When they all agree, confidence is high. The ensemble benefits from the models' *uncorrelated* errors — where XGBoost gets confused, Random Forest often gets it right.

### Handling Class Imbalance

FALSE POSITIVEs (50.6%) outnumber CANDIDATEs (20.7%) 2.4:1. Without correction, the model maximises accuracy by ignoring CANDIDATEs. We fix this with:

- **XGBoost:** `compute_sample_weight('balanced', y)` — upweights CANDIDATE samples so the model pays equal attention to all classes
- **HistGradBoost:** `class_weight='balanced'`
- **Random Forest:** `class_weight='balanced_subsample'` (resamples weights per tree, reducing variance)

In [ ]:
# Encode target and split
le = LabelEncoder()
y  = le.fit_transform(df[TARGET])
X  = df[ALL_FEATURES].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

n_classes = len(le.classes_)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"Classes:      {list(le.classes_)}")
print()

# Show that stratified split preserved class ratios
for cls_idx, cls_name in enumerate(le.classes_):
    train_ratio = (y_train == cls_idx).mean()
    test_ratio  = (y_test  == cls_idx).mean()
    print(f"  {cls_name:16s}: train={train_ratio:.1%}  test={test_ratio:.1%}")

In [ ]:
# ── BalancedXGBClassifier ─────────────────────────────────────────────────────
# XGBoost doesn't support class_weight like sklearn does.
# Instead we subclass it to auto-compute balanced sample weights at fit() time.
# This also avoids sklearn's metadata routing issue in VotingClassifier.

class BalancedXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        kwargs.setdefault('sample_weight', compute_sample_weight('balanced', y))
        return super().fit(X, y, **kwargs)


xgb = BalancedXGBClassifier(
    n_estimators=500,
    learning_rate=0.04,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multi:softprob',
    num_class=n_classes,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

hgb = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.04,
    max_depth=8,
    min_samples_leaf=20,
    l2_regularization=0.1,
    class_weight='balanced',
    random_state=SEED,
)

# Random Forest can't handle NaN, so we wrap it with a median imputer
rf_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(
        n_estimators=400,
        max_depth=16,
        min_samples_leaf=3,
        max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=SEED,
        n_jobs=-1,
    )),
])

model = VotingClassifier(
    estimators=[('xgb', xgb), ('hgb', hgb), ('rf', rf_pipe)],
    voting='soft',   # average probabilities, not majority vote
    n_jobs=1,
)

print("Model architecture ready.")
print("  - XGBoost        (500 trees, balanced sample weights)")
print("  - HistGradBoost  (500 iterations, class_weight=balanced)")
print("  - Random Forest  (400 trees, balanced_subsample) + median imputer")
print("  - Voting: soft (average predicted probabilities)")

---
## Section 8: Training

In [ ]:
import time
print("Training ensemble... (takes 3-6 minutes on Colab CPU)")
t0 = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"Done in {elapsed/60:.1f} minutes.")

---
## Section 9: Evaluation

We evaluate on the held-out test set — 1,913 KOIs the model has never seen.

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc          = accuracy_score(y_test, y_pred)
macro_f1     = f1_score(y_test, y_pred, average='macro')
weighted_f1  = f1_score(y_test, y_pred, average='weighted')
macro_recall = recall_score(y_test, y_pred, average='macro')
macro_prec   = precision_score(y_test, y_pred, average='macro')

print('=' * 52)
print(f'  Accuracy          : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  Macro F1          : {macro_f1:.4f}')
print(f'  Weighted F1       : {weighted_f1:.4f}')
print(f'  Macro Recall      : {macro_recall:.4f}')
print(f'  Macro Precision   : {macro_prec:.4f}')
print('=' * 52)
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# ── 9.1 Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))

# Normalise for display (but show raw counts in cells)
im = ax.imshow(cm, cmap='Blues', aspect='auto')
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(le.classes_, rotation=15, ha='right')
ax.set_yticklabels(le.classes_)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Held-Out Test Set (n=1,913)', pad=12)

for i in range(n_classes):
    for j in range(n_classes):
        pct = cm[i, j] / cm[i].sum() * 100
        color = 'white' if cm[i, j] > cm.max() * 0.5 else '#e6edf3'
        ax.text(j, i, f'{cm[i, j]}\n({pct:.0f}%)',
                ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print("Reading the matrix: rows = actual class, columns = predicted class.")
print("Diagonal = correct predictions. Off-diagonal = errors.")
print(f"\nCANDIDATE recall is lowest ({cm[0,0]/cm[0].sum()*100:.0f}%) — expected, since")
print("CANDIDATE literally means 'uncertain' even to human experts.")

In [ ]:
# ── 9.2 One-vs-Rest ROC curves ────────────────────────────────────────────────
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 6))
for i, (cls, color) in enumerate(CLASS_COLORS.items()):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{cls}  (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.03, color='grey')
ax.set_xlabel('False Positive Rate  (1 − Specificity)')
ax.set_ylabel('True Positive Rate  (Recall)')
ax.set_title('One-vs-Rest ROC Curves', pad=12)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("AUC interpretation: 1.0 = perfect, 0.5 = random guessing.")
print("All three classes are well above 0.5, showing the model genuinely discriminates.")

In [ ]:
# ── 9.3 Confidence calibration — are the probabilities trustworthy? ───────────
# When the model says 80% probability, does it actually get ~80% of those right?

# Get the predicted probability for the predicted class (max probability)
predicted_proba = y_proba.max(axis=1)
correct = (y_pred == y_test)

bins = np.linspace(0.33, 1.0, 10)
bin_centers, accuracies, counts = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (predicted_proba >= lo) & (predicted_proba < hi)
    if mask.sum() > 10:
        bin_centers.append((lo + hi) / 2)
        accuracies.append(correct[mask].mean())
        counts.append(mask.sum())

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0.33, 1.0], [0.33, 1.0], 'k--', alpha=0.4, label='Perfect calibration')
sc = ax.scatter(bin_centers, accuracies, c=counts, cmap='YlOrRd', s=80, zorder=5)
ax.plot(bin_centers, accuracies, color='#4fc3f7', lw=2)
plt.colorbar(sc, ax=ax, label='Number of samples in bin')
ax.set_xlabel('Predicted confidence (max class probability)')
ax.set_ylabel('Actual accuracy in that confidence bin')
ax.set_title('Confidence Calibration', pad=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("A well-calibrated model follows the diagonal — when it says 80%, it's right ~80% of the time.")
print("Deviations show where to apply Platt scaling or isotonic regression for better calibration.")

---
## Section 10: Cross-Validation

A single train/test split could be lucky or unlucky. Cross-validation runs the full training + evaluation procedure multiple times on different splits to get a reliable estimate of true performance.

We use **5-fold stratified CV** — each fold keeps the same class ratio as the full dataset, so no fold accidentally has too few CANDIDATEs.

In [ ]:
print("Running 5-fold stratified cross-validation...")
print("(Each fold retrains the full ensemble — takes ~15-20 minutes on Colab)")
print()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=1)

print(f"Per-fold Macro F1: {['%.4f' % s for s in cv_scores]}")
print(f"Mean:  {cv_scores.mean():.4f}")
print(f"Std:   {cv_scores.std():.4f}")
print()
print(f"The low standard deviation (±{cv_scores.std():.4f}) confirms the model is stable")
print("and not just fitting one lucky split.")

# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 6), cv_scores, color='#4fc3f7', alpha=0.8, edgecolor='none')
ax.axhline(cv_scores.mean(), color='#ffcc80', lw=2, linestyle='--', label=f'Mean = {cv_scores.mean():.4f}')
ax.fill_between([-0.5, 5.5],
                [cv_scores.mean() - cv_scores.std()] * 2,
                [cv_scores.mean() + cv_scores.std()] * 2,
                alpha=0.15, color='#ffcc80', label=f'±1 std = {cv_scores.std():.4f}')
ax.set_xticks(range(1, 6))
ax.set_xticklabels([f'Fold {i}' for i in range(1, 6)])
ax.set_ylabel('Macro F1 Score')
ax.set_title('5-Fold Cross-Validation Results', pad=12)
ax.set_ylim(0.7, 0.85)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 11: SHAP Explainability

SHAP (SHapley Additive exPlanations) answers the question: **for a given prediction, how much did each feature contribute?**

SHAP is rooted in game theory — it fairly distributes the model's output among the input features, like splitting credit in a team effort. It gives us both global feature importance (averaged over many predictions) and local explanations (for one specific KOI).

In [ ]:
print("Computing SHAP values using XGBoost sub-model...")
bg = X_train.sample(min(500, len(X_train)), random_state=SEED)
explainer = shap.TreeExplainer(model.estimators_[0])  # XGBoost
sv = np.array(explainer.shap_values(bg))

# XGBoost ≥2.0 returns shape (n_samples, n_features, n_classes)
# Average absolute SHAP across samples and classes
if sv.ndim == 3:
    mean_shap = np.abs(sv).mean(axis=(0, 2))
elif sv.ndim == 2:
    mean_shap = np.abs(sv).mean(axis=0)
else:
    mean_shap = np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)

feat_imp = sorted(zip(ALL_FEATURES, mean_shap.tolist()), key=lambda x: x[1], reverse=True)
top_k = 15
top_names = [f for f, _ in feat_imp[:top_k]]
top_vals  = [v for _, v in feat_imp[:top_k]]

print(f"\nTop {top_k} features by SHAP importance:")
for name, val in feat_imp[:top_k]:
    tag = ' ← ENGINEERED' if name in ENGINEERED else ''
    print(f"  {name:35s}  {val:.4f}{tag}")

In [ ]:
# ── 11.1 Global feature importance bar chart ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

colors = ['#4fc3f7' if n in ENGINEERED else '#a5d6a7' for n in top_names]
bars = ax.barh(top_names[::-1], top_vals[::-1], color=colors[::-1], edgecolor='none')

for bar, val in zip(bars, top_vals[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8.5, color='#e6edf3')

legend_elements = [
    mpatches.Patch(color='#4fc3f7', label='Engineered feature'),
    mpatches.Patch(color='#a5d6a7', label='Raw Kepler measurement'),
]
ax.legend(handles=legend_elements, loc='lower right')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title(f'Top {top_k} Features — Global SHAP Importance (XGBoost)', pad=12)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

eng_in_top5 = sum(1 for n in top_names[:5] if n in ENGINEERED)
print(f"{eng_in_top5} of the top 5 features are engineered — proving the feature")
print("engineering added real predictive value, not just noise.")

In [ ]:
# ── 11.2 SHAP waterfall for one prediction ────────────────────────────────────
# Pick one sample from each class to explain
for cls_idx, cls_name in enumerate(le.classes_):
    class_mask = y_test == cls_idx
    if class_mask.sum() == 0:
        continue
    sample_idx = np.where(class_mask)[0][0]
    sample_X = X_test.iloc[[sample_idx]]
    pred = le.classes_[model.predict(sample_X)[0]]
    proba = model.predict_proba(sample_X)[0]
    print(f"\nExample {cls_name} KOI:")
    print(f"  Predicted: {pred}")
    print(f"  Probabilities: CANDIDATE={proba[0]:.3f}  CONFIRMED={proba[1]:.3f}  FALSE POS={proba[2]:.3f}")

print("\n(Full SHAP waterfall plots require shap.plots.waterfall with Explanation objects)")
print("See the Celesta web app's Performance section for interactive feature importance.")

---
## Section 12: What the Model Learned

### The Most Important Signal: Multi-Event Statistic (`koi_max_mult_ev`)

The highest-importance feature is the **multiple event statistic** — a measure of how consistently the transit repeats across many orbits. A real planet transits with clockwork precision: every 2.2 days (or 365 days, or whatever its period is), the star dims by exactly the same amount. The multi-event statistic accumulates this repeating signal and grows with the square root of the number of transits.

A false positive — an eclipsing binary, a cosmic ray, or an instrument glitch — doesn't repeat consistently. Its multi-event statistic stays low even if one transit looked spectacular.

### The Key Ratio: `single_multi_ratio` (Engineered Feature #4)

This ratio captures a critical asymmetry: when a signal spikes on one transit but doesn't repeat, the single-event stat is high and the multi-event stat is relatively low. The ratio is high. This is the fingerprint of a false positive.

For a real planet, both stats should scale together — the ratio stays in a predictable range.

### Why CANDIDATE is the Hardest Class

CANDIDATE is literally NASA's way of saying *"we don't know yet."* These KOIs passed automated screening (so they look somewhat planet-like) but haven't been confirmed or ruled out by follow-up observations. The model cannot know what the follow-up will reveal — no ML model can. What it does know is that CANDIDATE KOIs sit in the ambiguous overlap between the other two classes, which is exactly why they need follow-up observations. The model's 64.2% F1 on CANDIDATEs is not a failure — it correctly reflects genuine scientific uncertainty.

### Stellar Metallicity (`koi_smet`) — A Surprise Finding

Stellar metallicity ranked #7 in SHAP importance. This is astrophysically meaningful: metal-rich stars are significantly more likely to host planets (the planet formation process is more efficient when the protoplanetary disk has more heavy elements). The model picked up this real statistical correlation from the training data without being told about it.

### Leakage-Free Guarantee

Every feature in the model is a raw physical measurement from the Kepler photometer or a stellar spectroscopy survey. None of them are:
- NASA's own vetting scores
- Automated classifier outputs  
- Human review flags
- Disposition probabilities

This is the same information an astrophysicist would start with before touching a telescope.

---
## Section 13: Conclusion

### What We Built

Celesta is a leakage-free, production-deployed machine learning classifier for the NASA Kepler exoplanet dataset. Starting from raw telescope photometry and stellar measurements, it:

1. **Cleans and prepares** 9,564 KOI records with principled missing-value handling
2. **Engineers 8 physics-motivated features** from transit geometry, Kepler's laws, and signal statistics
3. **Trains a soft-voting ensemble** of XGBoost + HistGradientBoosting + Random Forest, handling class imbalance with balanced weights
4. **Achieves 81.3% accuracy and 78.8% macro F1** on unseen data, validated with 5-fold CV (78.6% ± 0.7%)
5. **Explains every prediction** with SHAP values, pointing to the most physically meaningful features
6. **Deploys as a full web application** with 3D visualisation, live predictions via REST API, and a celestial object explorer

### Results Summary

| Metric | Value |
|---|---|
| Accuracy | **81.3%** |
| Macro F1 | **78.8%** |
| CV Macro F1 | **78.6% ± 0.7%** |
| CANDIDATE F1 | 64.2% |
| CONFIRMED F1 | 86.6% |
| FALSE POSITIVE F1 | 85.7% |

### What Could Be Improved

- **Bayesian hyperparameter tuning** (Optuna) — the current hyperparameters were set by hand
- **ADASYN oversampling** specifically for the CANDIDATE class
- **Probability calibration** (Platt scaling or isotonic regression) — would make confidence scores more reliable
- **TabNet or shallow MLP** as a fourth ensemble member — neural approaches sometimes catch non-linear patterns that tree ensembles miss
- **Full Kepler DR25 TCE dataset** — includes pipeline-level features not in the KOI cumulative table

### Acknowledgements

- **Dataset:** NASA Exoplanet Archive, Kepler Objects of Interest Cumulative Table (public domain)
- **SHAP library:** Lundberg & Lee, 2017 — [A Unified Approach to Interpreting Model Predictions](https://arxiv.org/abs/1705.07874)
- **XGBoost:** Chen & Guestrin, 2016 — [XGBoost: A Scalable Tree Boosting System](https://arxiv.org/abs/1603.02754)
- **Challenge:** India High School Exoplanet Data Challenge / Celesta

---

*Made by Srinath V Venkateshan — vvsrinath0@gmail.com*